In [ ]:
# Cell 0 — BLAS threading
import os
os.environ['OMP_NUM_THREADS']     = '1'
os.environ['MKL_NUM_THREADS']     = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

In [ ]:
# Cell 1 — imports
%load_ext autoreload
%autoreload 2

import pickle
import numpy as np
import sys
sys.path.append('../../..')
sys.path.append('../../../results')

from utils import *

In [ ]:
# =====================================================================
# Normal matrix construction
# =====================================================================
def normal_W(Lambda, Q, tol=1e-8):
    """
    Construct a real normal matrix with the same eigenspectrum as a PATNs
    solution, using the same Q but setting the upper-triangular part to zero.

    Parameters
    ----------
    Lambda : np.ndarray, shape (N,) complex — eigenvalues (with conjugate pairs)
    Q      : np.ndarray, shape (N, N)       — orthonormal Schur basis
    tol    : float — threshold for identifying real eigenvalues

    Returns
    -------
    W_normal : np.ndarray, shape (N, N), real
    """
    Lambda = np.asarray(Lambda)
    N = Lambda.size

    real_vals   = Lambda[np.abs(Lambda.imag) < tol].real
    complex_pos = Lambda[Lambda.imag > tol]

    if real_vals.size + 2 * complex_pos.size != N:
        print(f'Warning: spectrum pairing mismatch: '
              f'{real_vals.size} real, {complex_pos.size} complex pairs, N={N}')

    B = np.zeros((N, N), dtype=float)
    k = 0
    for a in real_vals:
        B[k, k] = a
        k += 1
    for lam in complex_pos:
        a, b = lam.real, lam.imag
        B[k:k+2, k:k+2] = np.array([[a, -b], [b, a]])
        k += 2

    W_normal = Q @ B @ Q.conj().T
    if not np.iscomplexobj(Q):
        W_normal = W_normal.real
    return W_normal

In [ ]:
# Cell 2 — load data
# Download from Zenodo [DOI] and place under data/fig2/
data_root = '../../../data/fig2'
sigmas    = ['sigma02', 'sigma05', 'sigma1']

solutions = []
for s in sigmas:
    with open(f'{data_root}/{s}/Ws.pkl', 'rb') as f:
        data = pickle.load(f)
    solutions.append([d for d in data if not np.all(d.get('W') == 0)])

solutions0, solutions1, solutions2 = solutions

In [ ]:
# Cell 3 — create and save normal matrices for each sigma
# WARNING: do not run unless you need to regenerate the normal matrices

for sigma_idx, (sol_list, sigma_key) in enumerate(zip(solutions, sigmas)):
    normal_Ws = []
    for sol in sol_list:
        W_norm = normal_W(sol['Lambda'], sol['Q'])
        normal_Ws.append({
            'Lambda'  : sol['Lambda'],
            'Q'       : sol['Q'],
            'W_normal': W_norm
        })
    out_path = f'{data_root}/{sigma_key}/Ws_normal.pkl'
    with open(out_path, 'wb') as f:
        pickle.dump(normal_Ws, f)
    print(f'Saved {out_path} ({len(normal_Ws)} matrices)')